# Lesson 7, one call at a time

Lesson 7 built a decision tree from Gini impurity, turned its depth like a dial,
watched one tree change with its sample, and then combined many trees three ways:
bagging, the random forest and gradient boosting. This notebook takes each step on
its own, on a few loans where a pencil can follow it and on the lesson's own 1,200
loans where the lesson's numbers come from, so that the slide's number and the call
that makes it sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb` explains `fit`,
`predict`, `predict_proba`, `random_state` and `Pipeline`; this one assumes them.

Every atom names the slide it comes from, by title, and lesson 7's notebooks treat
each idea at full scale. The whole notebook runs in about a minute.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (BaggingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

# Lesson 7's data: 1,200 loan applicants, income (k EUR) and debt ratio, and whether
# the loan defaulted; 7% of the labels were flipped after the rule was applied.
sys.path.insert(0, str(Path.cwd().parent / "Lessons" / "07_trees_and_ensembles" / "Notebooks"))
from loan_data import flipped_labels, load_loans, load_with_noise

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

X, y = load_loans()
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # the lesson's five folds
print(f"{len(X)} applicants, {int(y.sum())} defaulted, columns: {list(X.columns)}")

1200 applicants, 464 defaulted, columns: ['income_k', 'debt_ratio']


## 1. Gini impurity by hand, against the tree's own number

A group's Gini impurity is $1 - \sum_c p_c^2$; for two classes, $2p(1-p)$. Compute it for the 1,200 applicants and for the two groups the debt ceiling makes, then read the same three numbers from a one-split tree.

*From:* slides *Gini impurity*; *For two classes, one number*; handout section 2.2.

### Your prediction, before you run the next cell

The whole group is 39% defaulted: is its Gini closer to 0 or to 0.5? Will the group above the ceiling, 94% defaulted, be purer or less pure than the whole?

In [2]:
def gini(labels):
    p = np.mean(labels)                     # share defaulted
    return 1 - (p ** 2 + (1 - p) ** 2)

above = X["debt_ratio"] > 0.82
print(f"all 1,200:           {y.mean():.3f} defaulted, Gini {gini(y):.3f}")
print(f"debt <= 0.82 ({(~above).sum()}):  {y[~above].mean():.3f} defaulted, Gini {gini(y[~above]):.3f}")
print(f"debt  > 0.82 ({above.sum()}):   {y[above].mean():.3f} defaulted, Gini {gini(y[above]):.3f}")

stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print("\nthe tree's own numbers, root then left then right:", stump.tree_.impurity.round(3))
print("and its sizes:", stump.tree_.n_node_samples)

all 1,200:           0.387 defaulted, Gini 0.474
debt <= 0.82 (971):  0.256 defaulted, Gini 0.381
debt  > 0.82 (229):   0.939 defaulted, Gini 0.115

the tree's own numbers, root then left then right: [0.474 0.381 0.115]
and its sizes: [1200  971  229]


**What to look at.** Gini by hand and the tree's own `tree_.impurity` agree: 0.474 for all 1,200, 0.381 below the debt ceiling, 0.115 above it. The whole group, 39% defaulted, is close to the maximum of 0.5 - very mixed; the 229 above the ceiling, 94% defaulted, are close to pure. The sizes, 971 and 229, are `tree_.n_node_samples`.

**Try changing.** compute the Gini of the 971 split again on income at 28, and check it against a depth-2 tree's `tree_.impurity`.

## 2. The gain of the root split, by counting

The gain is the parent's Gini minus the children's, each child weighted by its share of the applicants. Count, then compute; then make the predictable slip on purpose.

*From:* slides *The gain a split buys*; *Worked example: the root split*; handout section 2.2.

### Your prediction, before you run the next cell

Weighted average or plain average of the two children: which one does the tree use, and which is larger here?

In [3]:
n_left, n_right = (~above).sum(), above.sum()
g_parent, g_left, g_right = gini(y), gini(y[~above]), gini(y[above])
weighted = n_left / 1200 * g_left + n_right / 1200 * g_right
print(f"weighted children: {n_left}/1200 x {g_left:.3f} + {n_right}/1200 x {g_right:.3f} = {weighted:.3f}")
print(f"gain = {g_parent:.3f} - {weighted:.3f} = {g_parent - weighted:.4f}")
print(f"the plain average instead: ({g_left:.3f} + {g_right:.3f}) / 2 = {(g_left + g_right) / 2:.3f}")

t = stump.tree_
print(f"\nfrom the tree: {t.impurity[0]:.4f} - ({t.n_node_samples[1]}/{t.n_node_samples[0]} x {t.impurity[1]:.4f}"
      f" + {t.n_node_samples[2]}/{t.n_node_samples[0]} x {t.impurity[2]:.4f})"
      f" = {t.impurity[0] - (t.n_node_samples[1] * t.impurity[1] + t.n_node_samples[2] * t.impurity[2]) / t.n_node_samples[0]:.4f}")

weighted children: 971/1200 x 0.381 + 229/1200 x 0.115 = 0.330
gain = 0.474 - 0.330 = 0.1438
the plain average instead: (0.381 + 0.115) / 2 = 0.248

from the tree: 0.4743 - (971/1200 x 0.3814 + 229/1200 x 0.1148) = 0.1438


**What to look at.** The tree weighs each child by its share of the applicants: 971/1200 x 0.381 + 229/1200 x 0.115 = 0.330, and the gain is 0.474 - 0.330 = 0.1438, the number in slide 8's root box. The plain average, 0.248, would give the small, pure group of 229 as much say as the 971 and overstate the gain.

**Try changing.** compute the gain of the split on income at 28 on all 1,200, and confirm that it is smaller than 0.1438 - which is why the root asks about debt.

## 3. Thresholds: infinitely many, but few different splits

On eight loans, every midpoint between two neighbouring debt ratios is a candidate threshold. Any threshold between the same two neighbours makes the same two groups.

*From:* slides *Scanning for the best split*; *Every threshold, scored*; handout section 2.2.

### Your prediction, before you run the next cell

Eight loans with eight different debt ratios: how many different splits on debt are there? Will thresholds 0.50 and 0.55 give the same groups if no loan lies between them?

In [4]:
few = X.iloc[:8].copy()
few["defaulted"] = y.iloc[:8].values
few = few.sort_values("debt_ratio")
values = few["debt_ratio"].to_numpy()
midpoints = (values[:-1] + values[1:]) / 2
print(few.to_string())
print("\ncandidate thresholds (midpoints):", midpoints.round(4), f"-> {len(midpoints)} different splits")

lo, hi = values[3], values[4]
for threshold in (lo + 0.25 * (hi - lo), (lo + hi) / 2, lo + 0.75 * (hi - lo)):
    left = few.index[few["debt_ratio"] <= threshold].tolist()
    print(f"threshold {threshold:.4f}: left group {left}")

   income_k  debt_ratio  defaulted
4     27.53      0.0266          1
5     98.05      0.0495          0
7     82.89      0.4862          0
1     55.11      0.5509          0
0     81.92      0.5664          1
6     80.89      0.6016          1
3     75.79      0.7105          1
2     88.69      0.8285          1

candidate thresholds (midpoints): [0.038  0.2679 0.5186 0.5586 0.584  0.656  0.7695] -> 7 different splits
threshold 0.5548: left group [4, 5, 7, 1]
threshold 0.5586: left group [4, 5, 7, 1]
threshold 0.5625: left group [4, 5, 7, 1]


**What to look at.** Eight loans with eight different debt ratios give seven midpoints, so seven different splits on debt, however many thresholds one could write. The three thresholds between 0.5509 and 0.5664 all send the same four loans to the left: any threshold in the same gap makes the same split, and the tree reports the midpoint, 0.5586.

**Try changing.** add the income column: how many candidate splits does the tree score in all, on eight loans?

## 4. A tree on ten loans: the rules, and one applicant's path

Fit a depth-2 tree on ten loans, print it as if/else rules, and follow one new applicant down the tree.

*From:* slides *The algorithm, drawn*; *The loop, in seven lines*; *Reading a tree, and the knob that keeps it small*; handout sections 2.1 and 4.

### Your prediction, before you run the next cell

A depth-2 tree asks at most how many questions of one applicant? Will the leaf it lands in answer with a probability, or with a class?

In [5]:
ten_X, ten_y = X.iloc[:10], y.iloc[:10]
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(ten_X, ten_y)
print(export_text(small, feature_names=list(X.columns)))

applicant = pd.DataFrame({"income_k": [45.0], "debt_ratio": [0.60]})
leaf = small.apply(applicant)[0]
path = small.decision_path(applicant).indices
print(f"the applicant visits nodes {list(path)} and lands in leaf {leaf}")
print(f"that leaf holds {small.tree_.n_node_samples[leaf]} of the ten; predict_proba "
      f"{small.predict_proba(applicant)[0].round(3)}, predict {small.predict(applicant)[0]}")

|--- debt_ratio <= 0.56
|   |--- income_k <= 28.89
|   |   |--- class: 1
|   |--- income_k >  28.89
|   |   |--- class: 0
|--- debt_ratio >  0.56
|   |--- class: 1

the applicant visits nodes [0, 4] and lands in leaf 4
that leaf holds 4 of the ten; predict_proba [0. 1.], predict 1


**What to look at.** The whole model is two questions and three leaves, printed as if/else rules. The applicant at income 45 and debt 0.60 is sent right at the root (debt above 0.56) and lands in leaf 4 at once: a path can be shorter than the depth. The leaf holds 4 of the ten loans, all defaulted, so `predict_proba` is 1 for default and `predict` returns the leaf's majority, 1. Training builds the questions; prediction only walks down them.

**Try changing.** give the applicant a debt ratio of 0.40 and an income of 25: which leaf, and why?

## 5. Depth against a budget of leaves

`max_depth` caps every path; `max_leaf_nodes` spends a budget of leaves where they gain most. Compare the two on the 1,200 loans: leaves, real depth, and the five-fold score.

*From:* slides *Depth is the bias-variance dial*; *Reading a tree, and the knob that keeps it small*; handout sections 3 and 4.

### Your prediction, before you run the next cell

A depth-8 tree and a 9-leaf tree: which has more leaves, and which scores better on the folds?

In [6]:
for name, model in (("max_depth=8", DecisionTreeClassifier(max_depth=8, random_state=0)),
                    ("max_leaf_nodes=9", DecisionTreeClassifier(max_leaf_nodes=9, random_state=0)),
                    ("no limit", DecisionTreeClassifier(random_state=0))):
    fitted = model.fit(X, y)
    scores = cross_val_score(model, X, y, cv=folds)
    leaf_sizes = fitted.tree_.n_node_samples[fitted.tree_.children_left < 0]
    print(f"{name:17s} {fitted.get_n_leaves():4d} leaves, real depth {fitted.get_depth():2d}, "
          f"median leaf {np.median(leaf_sizes):5.1f} applicants, one-applicant leaves "
          f"{(leaf_sizes == 1).sum():3d}, training {fitted.score(X, y):.3f}, "
          f"five folds {scores.mean():.3f}")

max_depth=8         71 leaves, real depth  8, median leaf   5.0 applicants, one-applicant leaves  24, training 0.951, five folds 0.882
max_leaf_nodes=9     9 leaves, real depth  7, median leaf 103.0 applicants, one-applicant leaves   0, training 0.908, five folds 0.901
no limit           156 leaves, real depth 24, median leaf   2.0 applicants, one-applicant leaves  72, training 1.000, five folds 0.852


**What to look at.** The depth-8 tree has 71 leaves, a median leaf of 5 applicants and 24 leaves holding one applicant each; the nine-leaf tree has 9 leaves, a median of 103 applicants and none of one, and still scores better on the five folds, 0.901 against 0.882. The unconstrained tree reaches training accuracy 1.000 with 156 leaves, 72 of them holding a single applicant, a real depth of 24, and the worst fold score, 0.852. A budget of leaves spends them where they gain most; a depth cap spends them evenly.

**Try changing.** try `max_leaf_nodes` 5, 15 and 30: where does the fold score stop improving?

## 6. The noise ceiling, and the flipped label a tree learns

7% of the labels were flipped after the rule was applied, so a model that follows the rule scores about 0.93. Training accuracy above that means flipped labels learned by heart.

*From:* slides *The curve bends downward*; *The same rule, three depths*; handout section 3.

### Your prediction, before you run the next cell

How many of the 1,200 labels were flipped? Will the unconstrained tree's training accuracy stop at 0.929 or go above it?

In [7]:
flipped = flipped_labels()
print(f"flipped labels: {flipped.sum()} of 1,200; the ceiling: 1 - {flipped.sum()}/1200 = {1 - flipped.mean():.3f}")
for depth in (2, 3, 8, None):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    right = tree.predict(X) == y.to_numpy()
    print(f"depth {str(depth):4s}: training accuracy {right.mean():.4f}, flipped labels "
          f"reproduced {(right & flipped.to_numpy()).sum()} of {flipped.sum()}")

flipped labels: 85 of 1,200; the ceiling: 1 - 85/1200 = 0.929
depth 2   : training accuracy 0.8400, flipped labels reproduced 12 of 85
depth 3   : training accuracy 0.8408, flipped labels reproduced 13 of 85
depth 8   : training accuracy 0.9508, flipped labels reproduced 38 of 85
depth None: training accuracy 1.0000, flipped labels reproduced 85 of 85


**What to look at.** 85 of the 1,200 labels were flipped, so the rule itself scores 1 - 85/1200 = 0.929. Going from depth 2 to depth 3 changes one prediction (0.8400 to 0.8408), and it is a flipped label. At depth 8 the tree reproduces 38 of the 85, and training accuracy 0.951 is already above the ceiling; the unconstrained tree reproduces all 85 - it has learned the noise by heart.

**Try changing.** find the flipped label depth 3 learns: the applicant whose verdict differs between the depth-2 and depth-3 trees.

## 7. Five folds, five scores: the mean and the spread by hand

`cross_val_score` returns one accuracy per fold. Their mean is what the slides report; their standard deviation is the plus-or-minus.

*From:* slides *Measured, at seven depths*; *Cross-validation, step by step: five groups of 240*; handout sections 3 and 5.1.

### Your prediction, before you run the next cell

Will the five fold accuracies of the depth-8 tree all lie within 0.01 of 0.882? numpy's `std` and the textbook's: which one is larger?

In [8]:
scores = cross_val_score(DecisionTreeClassifier(max_depth=8, random_state=0), X, y, cv=folds)
right = (scores * 240).round().astype(int)
print("right in each fold of 240:", right, "-> accuracies", scores.round(4))
mean = right.sum() / 1200
deviations = scores - mean
print(f"mean = {right.sum()} / 1200 = {mean:.4f}")
print(f"squared deviations {np.round(deviations ** 2, 6)}, sum {np.sum(deviations ** 2):.6f}")
print(f"divided by 5, square root: {np.sqrt(np.sum(deviations ** 2) / 5):.4f}  (np.std: {scores.std():.4f})")
print(f"divided by 4, square root: {np.sqrt(np.sum(deviations ** 2) / 4):.4f}  (np.std(ddof=1): {scores.std(ddof=1):.4f})")

right in each fold of 240: [210 211 215 214 209] -> accuracies [0.875  0.8792 0.8958 0.8917 0.8708]
mean = 1059 / 1200 = 0.8825
squared deviations [0.0001 0.     0.0002 0.0001 0.0001], sum 0.000465
divided by 5, square root: 0.0096  (np.std: 0.0096)
divided by 4, square root: 0.0108  (np.std(ddof=1): 0.0108)


**What to look at.** The five folds of 240 hold 210, 211, 215, 214 and 209 right: 1,059 of 1,200, a mean of 0.8825 (the slides' 0.882). The spread is the square root of the summed squared deviations divided by 5, 0.0096, which is what `np.std` returns; dividing by 4, as some statistics texts do, gives 0.0108, `np.std(ddof=1)`. The two conventions differ, and a report should say which one it uses.

**Try changing.** run the same with the unconstrained tree: is its spread larger or smaller than the depth-8 tree's?

## 8. Without replacement, with replacement

Slide 26's two trees used samples drawn without replacement: no one twice. A bootstrap sample draws with replacement: as many draws as rows, some rows twice, some never.

*From:* slides *A tree that doesn't stay still*; *Bagging: many trees, one vote*; *How much data one bootstrap sample sees*; handout sections 4 and 5.1.

### Your prediction, before you run the next cell

A bootstrap sample of the 1,200 loans has 1,200 draws: about how many different loans? And how many repeated draws, compared with the loans never drawn?

In [9]:
rng = np.random.default_rng(0)
without = rng.choice(1200, size=780, replace=False)
print(f"without replacement: {len(without)} draws, {len(np.unique(without))} different loans")

draws = rng.integers(0, 1200, size=1200)                  # with replacement
times = np.bincount(draws, minlength=1200)
print(f"with replacement:    {len(draws)} draws, {(times > 0).sum()} different loans, "
      f"{(times == 0).sum()} never drawn, {len(draws) - (times > 0).sum()} repeated draws")
print("loans drawn r times, r = 0..4:", np.bincount(times)[:5])
print(f"theory for the never drawn: 1200 x (1 - 1/1200)^1200 = {1200 * (1 - 1 / 1200) ** 1200:.1f}")

without replacement: 780 draws, 780 different loans
with replacement:    1200 draws, 753 different loans, 447 never drawn, 447 repeated draws
loans drawn r times, r = 0..4: [447 432 227  68  21]
theory for the never drawn: 1200 x (1 - 1/1200)^1200 = 441.3


**What to look at.** Without replacement, 780 draws are 780 different loans. With replacement, 1,200 draws give 753 different loans: 447 are never drawn - that tree's out-of-bag loans - and the repeated draws are also 447, because every repeat takes the place of a loan left out. Theory expects about 441 never drawn; one sample gives 447. And 432 loans are drawn once against 447 never: "once" and "never" are about equally common.

**Try changing.** draw five bootstrap samples with different seeds: how far does the count of loans never drawn move from 441?

## 9. Bagging by hand: five trees, their samples, and their vote

A `BaggingClassifier` of five trees on twelve loans. Look at what each tree drew, ask each tree about two new applicants, count the votes, and compare with `predict` and `predict_proba`.

*From:* slides *Bagging: many trees, one vote*; *Bagging, drawn*; handout section 5.

### Your prediction, before you run the next cell

Will `predict_proba` be the share of the five trees voting 'default'? Can two trees that saw different loans disagree on the same applicant?

In [10]:
twelve_X, twelve_y = X.iloc[:12], y.iloc[:12]
bag = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=5,
                        random_state=0).fit(twelve_X, twelve_y)
for k, rows in enumerate(bag.estimators_samples_):
    print(f"tree {k + 1} drew loans {sorted(rows.tolist())}  (never: {sorted(set(range(12)) - set(rows.tolist()))})")

new = pd.DataFrame({"income_k": [40.0, 75.0], "debt_ratio": [0.55, 0.30]})
votes = np.array([t.predict(new.to_numpy()[:, cols])
                  for t, cols in zip(bag.estimators_, bag.estimators_features_)])
print("\nvotes, one row per tree (1 = default):\n", votes)
print("share voting default:", votes.mean(axis=0), "| predict_proba:", bag.predict_proba(new)[:, 1],
      "| predict:", bag.predict(new))

tree 1 drew loans [0, 0, 2, 2, 3, 3, 6, 7, 7, 9, 9, 9]  (never: [1, 4, 5, 8, 10, 11])
tree 2 drew loans [0, 0, 1, 1, 3, 4, 5, 7, 8, 8, 9, 11]  (never: [2, 6, 10])
tree 3 drew loans [1, 1, 3, 4, 5, 5, 7, 7, 7, 8, 10, 11]  (never: [0, 2, 6, 9])
tree 4 drew loans [0, 0, 1, 3, 3, 3, 4, 4, 5, 5, 8, 11]  (never: [2, 6, 7, 9, 10])
tree 5 drew loans [1, 3, 3, 5, 5, 6, 6, 7, 9, 11, 11, 11]  (never: [0, 2, 4, 8, 10])

votes, one row per tree (1 = default):
 [[1 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]]
share voting default: [0.2 0. ] | predict_proba: [0.2 0. ] | predict: [0 0]


**What to look at.** Each tree drew 12 loans from 12, with repeats: tree 1 drew loans 0, 2, 3 and 7 twice and loan 9 three times, and never saw six loans. On the first applicant one tree says default and four say repaid; `predict_proba` is exactly the share voting default, 0.2, and `predict` is the majority, repaid. Trees that saw different loans can disagree on the same applicant; the vote settles it.

**Try changing.** raise `n_estimators` to 51: does the first applicant's share move, and does the verdict?

## 10. Out-of-bag by hand

Each loan is judged only by the trees whose sample missed it. Do it by hand for a 50-tree bagging on the 1,200 loans, and compare with `oob_score_`.

*From:* slides *Out-of-bag: each loan judged by the trees that never saw it*; *Out-of-bag against cross-validation*; handout section 5.1.

### Your prediction, before you run the next cell

With 50 trees, about how many judge each loan? Will the hand count equal `oob_score_` exactly?

In [11]:
bag50 = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=50,
                          oob_score=True, random_state=0).fit(X, y)
Xa = X.to_numpy()
missed = np.array([np.bincount(rows, minlength=1200) == 0 for rows in bag50.estimators_samples_])
tree_votes = np.array([t.predict(Xa[:, c]) for t, c in zip(bag50.estimators_, bag50.estimators_features_)])
voters = missed.sum(axis=0)
share = (tree_votes * missed).sum(axis=0) / voters
verdict = (share > 0.5).astype(int)                         # a tie goes to 0, repaid
print(f"trees judging each loan: {voters.mean():.1f} on average (0.368 x 50 = {0.368 * 50:.1f}), "
      f"from {voters.min()} to {voters.max()}")
print(f"loan 0: judged by trees {list(np.flatnonzero(missed[:, 0]) + 1)}, share voting default {share[0]:.2f}, "
      f"true label {y.iloc[0]}")
print(f"by hand: {(verdict == y.to_numpy()).sum()} of 1,200 right = {(verdict == y.to_numpy()).mean():.4f}"
      f" | oob_score_ = {bag50.oob_score_:.4f}")

trees judging each loan: 18.5 on average (0.368 x 50 = 18.4), from 7 to 30
loan 0: judged by trees [9, 11, 14, 16, 17, 19, 23, 31, 33, 38, 42, 45, 46, 48], share voting default 1.00, true label 1
by hand: 1085 of 1,200 right = 0.9042 | oob_score_ = 0.9042


**What to look at.** With 50 trees each loan is judged by 18.5 trees on average, close to 0.368 x 50 = 18.4, from 7 to 30. Loan 0, which defaulted, is judged by 14 trees, all saying default. Counting by hand, 1,085 of the 1,200 verdicts are right, 0.9042, exactly scikit-learn's `oob_score_`: nothing is set aside and nothing is trained again.

**Try changing.** set `n_estimators=10`: scikit-learn warns that some loans have no out-of-bag trees - why?

## 11. A random forest on two columns: one column per split

With $n = 2$ columns, `max_features='sqrt'` offers each split $\lfloor\sqrt{2}\rfloor = 1$ column, drawn at random at every split. Count the root questions of bagging's trees and the forest's.

*From:* slides *Random forests: trees less alike*; *Bagged trees against forest trees*; handout section 6.

### Your prediction, before you run the next cell

Out of 100 bagged trees, how many start on `debt_ratio`? And out of 100 forest trees?

In [12]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
bag100 = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=100,
                           random_state=0).fit(X_tr, y_tr)
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
print("columns offered per split in the forest:", forest.estimators_[0].max_features_)
for name, model, cols_of in (("bagging", bag100, bag100.estimators_features_),
                             ("forest", forest, [np.arange(2)] * 100)):
    roots = pd.Series([X.columns[c[t.tree_.feature[0]]] for t, c in zip(model.estimators_, cols_of)])
    alone = np.mean([(t.predict(X_te.to_numpy()[:, c]) == y_te.to_numpy()).mean()
                     for t, c in zip(model.estimators_, cols_of)])
    print(f"{name:8s} roots: {roots.value_counts().to_dict()}; one tree alone {alone:.3f}; "
          f"the vote {model.score(X_te, y_te):.3f}")

columns offered per split in the forest: 1
bagging  roots: {'debt_ratio': 100}; one tree alone 0.837; the vote 0.867
forest   roots: {'debt_ratio': 51, 'income_k': 49}; one tree alone 0.828; the vote 0.886


**What to look at.** Each split of the forest is offered one column. All 100 bagged trees start with the same question, on debt; the forest's start 51 on debt and 49 on income. A forest tree alone is a little worse, 0.828 against 0.837 on the test applicants, and the forest's vote is better, 0.886 against 0.867: what a vote needs is trees that err differently.

**Try changing.** set `max_features=None` in the forest: what happens to its roots, and to its vote?

## 12. Feature importance by hand, on the depth-2 tree

Each split's gain, weighted by its share of the applicants, is credited to the column it asks about; the sums are divided by their total.

*From:* slides *Feature importance on one tree*; *What a forest's importance says*; handout section 7.

### Your prediction, before you run the next cell

The root asks about debt and the left child about income: which column ends up with more than half of the importance?

In [13]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
t = tree2.tree_
credit = {"income_k": 0.0, "debt_ratio": 0.0}
for node in range(t.node_count):
    left, right = t.children_left[node], t.children_right[node]
    if left < 0:
        continue                                            # a leaf asks nothing
    n_p, n_l, n_r = t.n_node_samples[[node, left, right]]
    gain = t.impurity[node] - (n_l * t.impurity[left] + n_r * t.impurity[right]) / n_p
    column = X.columns[t.feature[node]]
    credit[column] += n_p / 1200 * gain
    print(f"node {node}: {column} <= {t.threshold[node]:.3f}, {n_p} applicants, gain {gain:.4f}, "
          f"credit {n_p / 1200 * gain:.4f}")
total = sum(credit.values())
print({c: round(v / total, 3) for c, v in credit.items()}, "| feature_importances_:",
      dict(zip(X.columns, tree2.feature_importances_.round(3))))

node 0: debt_ratio <= 0.820, 1200 applicants, gain 0.1438, credit 0.1438
node 1: income_k <= 28.060, 971 applicants, gain 0.0836, credit 0.0676
node 4: debt_ratio <= 0.947, 229 applicants, gain 0.0026, credit 0.0005
{'income_k': 0.319, 'debt_ratio': 0.681} | feature_importances_: {'income_k': 0.319, 'debt_ratio': 0.681}


**What to look at.** The root's credit is its gain times its share of the applicants, 0.1438 x 1; the left split's, 0.0836 x 971/1200 = 0.0676; the right split's, 0.0026 x 229/1200 = 0.0005. Debt collects 0.1443 and income 0.0676; divided by their total, debt 0.681 and income 0.319 - exactly `feature_importances_`.

**Try changing.** do it for a depth-3 tree: does income's share grow?

## 13. Permutation importance by hand: shuffle one column

On the forest with 20 columns of random numbers: shuffle one column of the held-out applicants, score again, and the accuracy lost is that column's importance.

*From:* slides *Permutation importance: the procedure*; *Permutation importance: claimed against measured*; handout section 7.1.

### Your prediction, before you run the next cell

Shuffling `debt_ratio` or shuffling a random column: which costs more accuracy? Will a random column cost exactly zero?

In [14]:
Xn, yn = load_with_noise(20)
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.3, random_state=0, stratify=yn)
noisy_forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xn_tr, yn_tr)
before = noisy_forest.score(Xn_te, yn_te)
rng = np.random.default_rng(3)
for column in ("debt_ratio", "income_k", "noise_00"):
    shuffled = Xn_te.copy()
    shuffled[column] = rng.permutation(shuffled[column].to_numpy())
    after = noisy_forest.score(shuffled, yn_te)
    print(f"shuffle {column:10s}: {before:.3f} -> {after:.3f}, drop {before - after:+.3f}")
print("\nthe forest's own importance on the 20 random columns:",
      f"{noisy_forest.feature_importances_[2:].sum():.1%}")

shuffle debt_ratio: 0.806 -> 0.600, drop +0.206
shuffle income_k  : 0.806 -> 0.708, drop +0.097
shuffle noise_00  : 0.806 -> 0.803, drop +0.003

the forest's own importance on the 20 random columns: 54.8%


**What to look at.** Shuffling the debt ratio among the 360 held-out applicants takes the accuracy from 0.806 to 0.600; shuffling income, to 0.708; shuffling a random column, to 0.803 - almost nothing. Yet the forest's own importance puts 54.8% on the twenty random columns. The importance a model reports is a claim; shuffling on unseen data is the check. `permutation_importance` does the same, repeating each shuffle and averaging.

**Try changing.** run `permutation_importance(noisy_forest, Xn_te, yn_te, n_repeats=20, random_state=0)` and compare its debt-ratio value with the single shuffle above.

## 14. Boosting by hand: the first two trees

Start from the mean; train a depth-2 regression tree on the residuals; add 0.3 times its prediction; repeat. Compare with `GradientBoostingRegressor` with two trees.

*From:* slides *Boosting, drawn: the first two steps*; *Building the ensemble, one tree at a time, for squared error*; handout section 8.

### Your prediction, before you run the next cell

After the first tree, does the prediction at $x = 5$ move all the way to the data, or only part of the way? Will scikit-learn's two-tree model give the same predictions as ours?

In [15]:
rng = np.random.default_rng(7)
x = np.sort(rng.uniform(0, 10, 200))
truth = np.where(x < 3, 1.0, np.where(x < 7, 3.5, 1.5)) + 0.6 * np.sin(x)
target = truth + rng.normal(0, 0.25, size=200)
Xt = x.reshape(-1, 1)

prediction = np.full(200, target.mean())
print(f"start: the mean, {target.mean():.3f}, for every x")
for k in (1, 2):
    tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(Xt, target - prediction)
    prediction = prediction + 0.3 * tree.predict(Xt)
    at5 = prediction[np.argmin(np.abs(x - 5))]
    print(f"after tree {k}: prediction near x = 5 is {at5:.3f}, mean |residual| {np.abs(target - prediction).mean():.3f}")

gbr = GradientBoostingRegressor(n_estimators=2, learning_rate=0.3, max_depth=2,
                                criterion="squared_error", random_state=0).fit(Xt, target)
print(f"\nlargest gap between our two-tree model and scikit-learn's: "
      f"{np.abs(gbr.predict(Xt) - prediction).max():.2e}")

start: the mean, 2.287, for every x
after tree 1: prediction near x = 5 is 2.578, mean |residual| 0.601
after tree 2: prediction near x = 5 is 2.781, mean |residual| 0.447

largest gap between our two-tree model and scikit-learn's: 0.00e+00


**What to look at.** The start is the mean, 2.287, for every x. After tree 1 the prediction near x = 5 moves to 2.578, only part of the way to the data around 3, because each correction is multiplied by 0.3; after tree 2, to 2.781. The average size of a residual falls from 0.601 to 0.447. scikit-learn's `GradientBoostingRegressor` with two trees gives exactly the same predictions: the largest gap is 0.

**Try changing.** set the learning rate to 1.0: where is the prediction at x = 5 after one tree?